# Experiment 8: Implementation of Naïve Bayes Classifier

**Course:** BCSE0731 — Machine Learning Lab  
**Aim:** Implement both:
1. **Gaussian Naïve Bayes** on continuous numerical features (`load_iris`), and
2. **Multinomial Naïve Bayes** with Laplace Smoothing (`alpha=1.0`) for **SMS/Email Spam Text Classification** using `CountVectorizer` / `TfidfVectorizer`.

### Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

### Part A: Gaussian Naïve Bayes on Continuous Features (Iris Dataset)

In [ ]:
iris = load_iris()
X_tr, X_te, y_tr, y_te = train_test_split(
    iris.data, iris.target, test_size=0.25, random_state=42, stratify=iris.target
)

gnb = GaussianNB()
gnb.fit(X_tr, y_tr)
y_pred_iris = gnb.predict(X_te)

print(f"GaussianNB Accuracy on Iris: {accuracy_score(y_te, y_pred_iris):.4f}")
print("\nClassification Report:\n", classification_report(y_te, y_pred_iris, target_names=iris.target_names))

ConfusionMatrixDisplay.from_predictions(y_te, y_pred_iris, display_labels=iris.target_names, cmap='Greens')
plt.title('GaussianNB Confusion Matrix (Iris)', fontweight='bold')
plt.show()

### Part B: Multinomial Naïve Bayes for Spam Filtering (Laplace Smoothing)

In [ ]:
# Self-contained Spam vs. Ham dataset
messages = [
    ("Win a free iPhone now! Claim your prize immediately", 1),
    ("Congratulations! You won $5000 cash reward click here", 1),
    ("Urgent offer! Limited time discount buy now free shipping", 1),
    ("FREE entry into weekly lottery! Text WIN to 80085", 1),
    ("Exclusive investment opportunity earn double money fast", 1),
    ("Get cheap loans approved instantly no credit check", 1),
    ("Claim your free gift voucher worth 1000 dollars today", 1),
    ("Hot deals waiting for you click link to claim bonus", 1),
    ("Hey, are we still meeting for the ML lab project today?", 0),
    ("Please review the attached lecture notes before tomorrow", 0),
    ("Can you send me the Python notebook from class?", 0),
    ("Dinner at 8 PM? Let me know if you are free", 0),
    ("The exam schedule has been posted on the university portal", 0),
    ("Happy birthday! Wishing you a wonderful year ahead", 0),
    ("Team standup is rescheduled to 10:30 AM in room 204", 0),
    ("Thanks for sharing the dataset and assignment guidelines", 0),
]

df_sms = pd.DataFrame(messages, columns=['text', 'label'])  # 1 = Spam, 0 = Ham
X_msg_tr, X_msg_te, y_msg_tr, y_msg_te = train_test_split(
    df_sms['text'], df_sms['label'], test_size=0.25, random_state=42, stratify=df_sms['label']
)

vec = CountVectorizer(stop_words='english')
X_vec_tr = vec.fit_transform(X_msg_tr)
X_vec_te = vec.transform(X_msg_te)

mnb = MultinomialNB(alpha=1.0)  # alpha=1.0 applies Laplace smoothing
mnb.fit(X_vec_tr, y_msg_tr)
y_msg_pred = mnb.predict(X_vec_te)

print(f"MultinomialNB Spam Accuracy: {accuracy_score(y_msg_te, y_msg_pred):.4f}")

# Test on brand-new unseen messages
new_msgs = [
    "Congratulations, claim your free bonus prize today!",
    "Let's study PCA and SVM together for tomorrow's quiz."
]
probs = mnb.predict_proba(vec.transform(new_msgs))
preds = mnb.predict(vec.transform(new_msgs))
for msg, p, pr in zip(new_msgs, preds, probs):
    label = "SPAM" if p == 1 else "HAM"
    print(f"[{label}] (P(Spam)={pr[1]:.3f}) -> '{msg}'")

### Conclusion
- **GaussianNB** models continuous features using class-conditional normal distributions.
- **MultinomialNB** with **Laplace smoothing (`alpha=1.0`)** handles unseen vocabulary words cleanly without zero-probability collapse.